In [1]:
# [Setup]: 
# Had to install miniconda (Windows) : https://www.anaconda.com/download/success
# Had to install Microsoft Visual Code, check Desktop development with C++ in install page : https://visualstudio.microsoft.com/visual-cpp-build-tools/
# Had to install Python extension in VSCodium

# Installs necessary packages. Ensure you have followed the steps above before continuing.
# Had to run in Anaconda prompt : 
    # conda create -n rcbplates python=3.11 -y
    # conda activate rcbplates
    # pip install ipympl
    # conda install -c conda-forge astropy photutils matplotlib numpy scipy pillow pycairo cairo pkg-config -y
    # # pip install daschlab astropy photutils matplotlib numpy scipy pillow pycairo cairo pkg-config astroquery
    # pip install daschlab

# Then, in VSCodium :
    # Ctrl + Shift + P -> Python: Select Interpreter -> Python 3.11 (rcbplates)


In [1]:
# NOTE : 01_Load_Cutout.ipynb - Cell 1

# This is an older version of the cutout loading algorithm, but it functions entirely to extract the cutouts for V CrA.

from IPython.display import HTML, display

display(HTML("""
<style>
.output, .output_text, .output_stream, .output_stdout,
.cell-output, .cell-output-print, .cell-output-stdout {
    color: white !important;
}
.output * { color: white !important; }
.output .ansi-yellow-fg, .output .ansi-yellow-foreground,
.warning, .Warning, .warnings {
    color: #FFD700 !important;
    text-shadow: 0px 0px 5px rgba(255, 215, 0, 0.3);
}
.output pre, .output code { color: white !important; }

/* Fix for log_output rendering as a white box: background_color is not a
   real ipywidgets Layout property, so it silently did nothing. This class
   + these selectors is the same approach Cell 2 uses successfully. */
.dark-log-output,
.dark-log-output .jp-OutputArea-output,
.dark-log-output .jp-OutputArea-child,
.dark-log-output .cell-output-ipywidget-background,
.dark-log-output .output_area,
.dark-log-output .jupyter-widgets-output-area,
.dark-log-output .widget-output,
.dark-log-output pre {
    background-color: #111 !important;
    color: white !important;
    border: none !important;
    box-shadow: none !important;
}
</style>
"""))

from daschlab import open_session
from pathlib import Path
from datetime import datetime
from concurrent.futures import ThreadPoolExecutor, wait, FIRST_COMPLETED
import threading
import time
import traceback
import ipywidgets as widgets
import pandas as pd

session_dir = Path(
    r"C:\Users\dapur\Downloads\Other\Research\rcb_Star_Dust_Survey\test_CNN\data\V_CrA"
)
session_dir.mkdir(parents=True, exist_ok=True)

cutout_dir = session_dir / "cutouts"
manifest_path = session_dir / "plate_manifest.csv"
summary_path = session_dir / "plate_manifest_summary.csv"

# Guard : won't let two download threads run at once if you accidentally
# re-run this cell while a previous download is still going.
if "_active_download_thread" in globals() and _active_download_thread.is_alive():
    print("⚠️  A download is already running in the background from a previous "
          "run of this cell. Click that run's Pause button and wait for "
          "'Paused' before re-running this cell.")
else:
    # Top bar : pause button + custom progress display, shown first so it
    # stays pinned at the top of this cell's output.
    pause_button = widgets.Button(
        description="⏸ Pause",
        button_style="warning",
        layout=widgets.Layout(width="120px", height="36px"),
    )
    progress_html = widgets.HTML(value="<i>Setting up session...</i>")

    top_bar = widgets.HBox(
        [pause_button, progress_html],
        layout=widgets.Layout(align_items="center", margin="0 0 12px 0"),
    )

    # Summary dashboard : sits right under the progress bar, always visible,
    # gets refreshed both before the run starts and again at the end.
    summary_html = widgets.HTML(value="")

    # background_color is not a valid ipywidgets Layout property (fixed via
    # add_class() + real CSS instead, same pattern as Cell 2).
    log_output = widgets.Output(
        layout=widgets.Layout(
            border="1px solid #444", padding="8px",
            max_height="300px", overflow="auto",
        )
    )
    log_output.add_class("dark-log-output")

    display(top_bar, summary_html, log_output)

    pause_event = threading.Event()

    def on_pause_clicked(b):
        pause_event.set()
        pause_button.description = "Pausing..."
        pause_button.disabled = True

    pause_button.on_click(on_pause_clicked)

    def format_eta(seconds):
        if seconds is None or seconds != seconds or seconds < 0:  # NaN/negative guard
            return "calculating..."
        m, s = divmod(int(seconds), 60)
        h, m = divmod(m, 60)
        return f"{h:d}:{m:02d}:{s:02d}" if h else f"{m:d}:{s:02d}"

    def render_progress(n_done, total, start_time, status_word="Downloading", color="#4CAF50"):
        elapsed = time.monotonic() - start_time
        pct = (n_done / total * 100) if total else 0
        rate = (n_done / elapsed) if elapsed > 0 and n_done > 0 else 0
        remaining = ((total - n_done) / rate) if rate > 0 else None

        bar_width = 380
        filled = int(bar_width * pct / 100)

        progress_html.value = f"""
        <div style="font-family: monospace; font-size: 14px; color: white; line-height: 1.6;">
          <div style="display:flex; align-items:center;">
            <div style="width:{bar_width}px; height:18px; background:#333;
                        border-radius:4px; overflow:hidden; margin-right:12px;">
              <div style="width:{filled}px; height:100%; background:{color};
                          transition: width 0.3s;"></div>
            </div>
            <b>{pct:5.1f}%</b>
          </div>
          <div style="margin-top:4px;">
            {status_word} &nbsp;
            <b>{n_done:,} / {total:,}</b> plates &nbsp;|&nbsp;
            <b>{rate:.2f}</b> plates/sec &nbsp;|&nbsp;
            Elapsed <b>{format_eta(elapsed)}</b> &nbsp;|&nbsp;
            ETA <b>{format_eta(remaining)}</b>
          </div>
        </div>
        """

    def render_summary_card(title, title_color, stat_items, footnote=None):
        """stat_items: list of (label, value, accent_color) tuples.
        value can be an int (formatted with commas) or a pre-formatted string."""
        cards = ""
        for label, value, accent in stat_items:
            display_value = f"{value:,}" if isinstance(value, int) else value
            cards += f"""
            <div style="background:#1a1a1a; border:1px solid {accent};
                        border-radius:8px; padding:10px 18px; margin:4px 8px 4px 0;
                        min-width:110px; text-align:center; flex:1;">
              <div style="font-size:22px; font-weight:bold; color:{accent};
                          font-family: monospace;">{display_value}</div>
              <div style="font-size:11px; color:#aaa; margin-top:3px;
                          text-transform:uppercase; letter-spacing:0.5px;">{label}</div>
            </div>
            """
        footnote_html = (
            f'<div style="color:#888; font-size:12px; margin-top:8px; font-family: monospace;">{footnote}</div>'
            if footnote else ""
        )
        summary_html.value = f"""
        <div style="background:#111; border:1px solid #333; border-radius:10px;
                    padding:12px 14px; margin-bottom:12px;">
          <div style="font-size:15px; font-weight:bold; color:{title_color};
                      font-family: monospace; margin-bottom:8px;">{title}</div>
          <div style="display:flex; flex-wrap:wrap;">{cards}</div>
          {footnote_html}
        </div>
        """

    def plate_limit_stats(df):
        """Returns (n_with_apass_limit, median_apass_limit, n_with_atlas_limit, median_atlas_limit)."""
        apass = pd.to_numeric(df.get("lim_mag_apass"), errors="coerce")
        atlas = pd.to_numeric(df.get("lim_mag_atlas"), errors="coerce")
        n_apass = int(apass.notna().sum())
        n_atlas = int(atlas.notna().sum())
        med_apass = f"{apass.median():.2f}" if n_apass else "--"
        med_atlas = f"{atlas.median():.2f}" if n_atlas else "--"
        return n_apass, med_apass, n_atlas, med_atlas

    def build_summary_df():
        """Builds a small, separate summary table -- NOT mixed into
        plate_manifest.csv, so that file stays clean (one row per exposure)
        for anything that does pd.read_csv() on it downstream, e.g. notebook 02."""
        counts = manifest_df["status"].value_counts().to_dict()
        n_apass, med_apass, n_atlas, med_atlas = plate_limit_stats(manifest_df)
        cutouts_now = sorted(cutout_dir.glob("*.fits")) if cutout_dir.exists() else []

        rows = [
            ("last_updated",              datetime.now().isoformat(timespec="seconds")),
            ("total_exposures",           len(manifest_df)),
            ("fits_files_on_disk",        len(cutouts_now)),
            ("downloaded",                counts.get("downloaded", 0)),
            ("unavailable",               counts.get("unavailable", 0)),
            ("error",                     counts.get("error", 0)),
            ("not_attempted",             counts.get("not_attempted", 0)),
            ("plates_with_lim_mag_apass", n_apass),
            ("median_lim_mag_apass",      med_apass),
            ("plates_with_lim_mag_atlas", n_atlas),
            ("median_lim_mag_atlas",      med_atlas),
        ]
        return pd.DataFrame(rows, columns=["metric", "value"])

    # Session + manifest setup
    with log_output:
        sess = open_session(str(session_dir))
        sess.select_target("V CrA")
        sess.select_refcat("apass")

        exposures = sess.exposures()
        print(f"Total exposures : {len(exposures)}")

        try:
            exposures_df = exposures.to_pandas()
        except Exception as e:
            print(f"Could not convert exposures table to pandas ({e}); manifest will have fewer columns.")
            exposures_df = pd.DataFrame(index=range(len(exposures)))

        exposures_df.insert(0, "exposure_index", range(len(exposures_df)))

        if manifest_path.exists():
            manifest_df = pd.read_csv(manifest_path)
            print(f"Loaded existing manifest with {len(manifest_df)} rows.")
        else:
            manifest_df = exposures_df.copy()
            manifest_df["status"] = "not_attempted"
            manifest_df["filename"] = ""
            manifest_df["error_message"] = ""
            manifest_df["last_updated"] = ""

        # lim_mag_apass / lim_mag_atlas are the plate limits (limiting
        # magnitude) for each exposure. They come from sess.exposures(),
        # not the cutout FITS header. At front of the CSV for visibility.
        front_cols = [c for c in [
            "exposure_index", "status", "filename",
            "lim_mag_apass", "lim_mag_atlas",
            "error_message", "last_updated",
        ] if c in manifest_df.columns]
        other_cols = [c for c in manifest_df.columns if c not in front_cols]
        manifest_df = manifest_df[front_cols + other_cols]

        manifest_df = manifest_df.set_index("exposure_index", drop=False)

        # Reconcile against what's actually on disk, in case a previous run
        # ended erronously and some rows never got saved as "downloaded".
        existing_files = {f.name for f in cutout_dir.glob("*.fits")} if cutout_dir.exists() else set()
        if existing_files and "filename" in manifest_df.columns:
            on_disk_mask = manifest_df["filename"].apply(
                lambda f: Path(f).name in existing_files if isinstance(f, str) and f else False
            )
            reconciled = (manifest_df["status"] != "downloaded") & on_disk_mask
            if reconciled.any():
                manifest_df.loc[reconciled, "status"] = "downloaded"
                print(f"Reconciled {reconciled.sum()} rows already on disk but not marked downloaded.")

        remaining_indices = manifest_df.index[manifest_df["status"] != "downloaded"].tolist()
        print(f"{len(remaining_indices)} cutouts remaining to download "
              f"(out of {len(manifest_df)} total).")

        # NOTE : TEST MODE ----------------------------------------------------------

        # Set to a small number (e.g. 10) to only download that many plates,
        # for testing. Set to None for the real full run.
        TEST_LIMIT = None

        if TEST_LIMIT is not None:
            remaining_indices = remaining_indices[:TEST_LIMIT]
            print(f"TEST MODE: only downloading {len(remaining_indices)} plates this run.")

        # NOTE : ---------------------------------------------------------------------

    # Show current manifest state up top, before any downloading happens.
    _counts_before = manifest_df["status"].value_counts().to_dict()
    _n_apass, _med_apass, _n_atlas, _med_atlas = plate_limit_stats(manifest_df)
    render_summary_card(
        title="Manifest Overview (before this run)",
        title_color="#90caf9",
        stat_items=[
            ("Total", len(manifest_df), "#90caf9"),
            ("Downloaded", _counts_before.get("downloaded", 0), "#4CAF50"),
            ("Unavailable", _counts_before.get("unavailable", 0), "#FFA726"),
            ("Errored", _counts_before.get("error", 0), "#EF5350"),
            ("Not attempted", _counts_before.get("not_attempted", 0), "#9E9E9E"),
            ("Plate limit (APASS)", f"{_n_apass:,} plates | med {_med_apass}", "#BA68C8"),
            ("Plate limit (ATLAS)", f"{_n_atlas:,} plates | med {_med_atlas}", "#4DD0E1"),
        ],
        footnote=f"{len(remaining_indices):,} plates queued for this run.",
    )

    total_remaining = max(len(remaining_indices), 1)
    render_progress(0, total_remaining, time.monotonic(), status_word="Starting...")

    # Conservative (DASCH's docs warn bulk cutout fetching loads their API server).
    MAX_WORKERS = 6

    def fetch_one(i):
        try:
            result = sess.cutout(i)
            if result is None:
                return i, "unavailable", None, "no cutout available (likely unscanned)"
            return i, "downloaded", str(result), None
        except Exception as e:
            return i, "error", None, str(e)

    def save_manifest():
        """Fault-tolerant save - e.g. won't crash the thread if the CSV is
        currently open in Excel (PermissionError on Windows). Logs a warning
        instead and just tries again next time."""
        try:
            manifest_df.reset_index(drop=True).sort_values("exposure_index").to_csv(
                manifest_path, index=False
            )
            return True
        except Exception as e:
            with log_output:
                print(f"Couldn't save manifest right now ({e}). "
                      f"If it's open in Excel, close it - will retry automatically.")
            return False

    def save_summary():
        """Writes the separate plate_manifest_summary.csv file. Also
        fault-tolerant, same reasoning as save_manifest()."""
        try:
            build_summary_df().to_csv(summary_path, index=False)
            return True
        except Exception as e:
            with log_output:
                print(f"Couldn't save summary right now ({e}). Will retry automatically.")
            return False

    def save_all():
        ok1 = save_manifest()
        ok2 = save_summary()
        return ok1 and ok2

    # Background download loop
    def run_downloads():
        counts = {"downloaded": 0, "unavailable": 0, "error": 0}
        n_done = 0
        start_time = time.monotonic()
        stopped_early = False

        ex = ThreadPoolExecutor(max_workers=MAX_WORKERS)
        futures = {ex.submit(fetch_one, i): i for i in remaining_indices}
        pending = set(futures.keys())

        try:
            while pending:
                if pause_event.is_set():
                    stopped_early = True
                    with log_output:
                        print("\n⏸ Pause requested - no new downloads will start.")
                        print("   A few in-flight downloads may finish in the background; that's fine.")
                    ex.shutdown(wait=False, cancel_futures=True)
                    break

                done, pending = wait(pending, timeout=0.2, return_when=FIRST_COMPLETED)
                for fut in done:
                    i, status, filename, err_msg = fut.result()
                    manifest_df.loc[i, "status"] = status
                    manifest_df.loc[i, "filename"] = filename or ""
                    manifest_df.loc[i, "error_message"] = err_msg or ""
                    manifest_df.loc[i, "last_updated"] = datetime.now().isoformat(timespec="seconds")
                    counts[status] += 1
                    n_done += 1

                if done:
                    render_progress(n_done, len(futures), start_time)
                    save_all()
            else:
                ex.shutdown(wait=True)

        except Exception as e:
            # Catches anything unexpected (network errors, widget update
            # failures, etc.) so the button never gets stuck on "Pausing..."
            stopped_early = True
            with log_output:
                print(f"\n Download thread hit an unexpected error: {e}")
                traceback.print_exc()

        finally:
            # This ALWAYS runs (even on a crash) so the UI never freezes.
            saved = save_all()
            if not saved:
                time.sleep(1.0)
                save_all()

            cutouts = sorted(cutout_dir.glob("*.fits")) if cutout_dir.exists() else []

            final_word = "Paused" if stopped_early else "Done"
            render_progress(n_done, len(futures), start_time,
                             status_word=final_word,
                             color="#FFA726" if stopped_early else "#4CAF50")
            pause_button.description = final_word
            pause_button.disabled = True

            # Final dashboard. Replaces the "before this run" card.
            _counts_after = manifest_df["status"].value_counts().to_dict()
            _n_apass2, _med_apass2, _n_atlas2, _med_atlas2 = plate_limit_stats(manifest_df)
            title_word = "⏸ Paused" if stopped_early else "Final Summary"
            title_color = "#FFA726" if stopped_early else "#4CAF50"
            render_summary_card(
                title=f"{title_word} - this run: "
                      f"{counts['downloaded']:,} downloaded, "
                      f"{counts['unavailable']:,} unavailable, "
                      f"{counts['error']:,} errored",
                title_color=title_color,
                stat_items=[
                    ("Total exposures", len(manifest_df), "#90caf9"),
                    ("On disk", len(cutouts), "#64B5F6"),
                    ("Downloaded", _counts_after.get("downloaded", 0), "#4CAF50"),
                    ("Unavailable", _counts_after.get("unavailable", 0), "#FFA726"),
                    ("Errored", _counts_after.get("error", 0), "#EF5350"),
                    ("Not attempted", _counts_after.get("not_attempted", 0), "#9E9E9E"),
                    ("Plate limit (APASS)", f"{_n_apass2:,} plates | med {_med_apass2}", "#BA68C8"),
                    ("Plate limit (ATLAS)", f"{_n_atlas2:,} plates | med {_med_atlas2}", "#4DD0E1"),
                ],
                footnote=(
                    ("Safe to close now. Rerun this cell anytime to pick back up. &nbsp;|&nbsp; "
                     if stopped_early else "")
                    + f"Manifest: {manifest_path}  |  Summary: {summary_path}"
                ),
            )

            with log_output:
                print(f"\n{'Paused' if stopped_early else 'Finished'} - see summary card above for full breakdown.")

    _active_download_thread = threading.Thread(target=run_downloads, daemon=True)
    _active_download_thread.start()

KeyboardInterrupt: 

In [ ]:
# 01_Load_Cutout.ipynb : Cell 2

# Plate cutout visualizer. Shows the cutout, the header, and pixel data for all downloaded plates.

from pathlib import Path
from io import BytesIO
from astropy.io import fits
from astropy.time import Time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML, Javascript

# Dark styling for the dropdown + toggles + table + kill any default borders/backgrounds/outlines
display(HTML("""
<style>

.dark-dropdown select {
    background-color: #1e1e1e !important;
    color: white !important;
    border: 1px solid #555 !important;
}

.dark-dropdown select option {
    background-color: #1e1e1e !important;
    color: white !important;
}

.dark-toggle {
    color: white !important;
    background-color: #333 !important;
    border: 1px solid #777 !important;
    font-weight: bold !important;
    font-size: 13px !important;
    padding: 6px 10px !important;
}

.dark-toggle:hover {
    background-color: #4a4a4a !important;
}

/* Applied on top of dark-toggle when a toggle is switched on, for visibility */
.dark-toggle-active {
    background-color: #2e7d32 !important;
    border: 1px solid #66bb6a !important;
}

.dark-toggle-active:hover {
    background-color: #388e3c !important;
}

/* Search box styling, matches dark-dropdown look */
.dark-search input {
    background-color: #1e1e1e !important;
    color: white !important;
    border: 1px solid #555 !important;
    font-family: monospace !important;
}

.dark-search input::placeholder {
    color: #888 !important;
}

.dark-output,
.dark-output .output,
.dark-output .widget-output,
.jp-OutputArea-output,
.jp-OutputArea-child,
.cell-output-ipywidget-background,
.output,
.output_wrapper,
.widgetarea,
.jp-RenderedHTMLCommon,
.jupyter-widgets {
    background-color: #111 !important;
    border: none !important;
    box-shadow: none !important;
    outline: none !important;
}

body, .jp-Notebook, .jp-WindowedPanel-outer, .jp-Cell-outputArea {
    background-color: #111 !important;
}

.dark-table {
    border-collapse: collapse;
    color: #ddd;
    background-color: #111;
    font-family: monospace;
    font-size: 11px;
}

.dark-table th,
.dark-table td {
    border: 1px solid #333;
    padding: 2px 6px;
    text-align: right;
}

.dark-table th {
    background-color: #222;
    color: white;
}

.table-scroll-box {
    max-height: 500px;
    max-width: 100%;
    overflow: auto;
    background-color: #111;
}

.header-table {
    border-collapse: collapse;
    color: #ddd;
    background-color: #111;
    font-family: monospace;
    font-size: 12px;
    width: 100%;
}

.header-table th,
.header-table td {
    border: 1px solid #333;
    padding: 4px 10px;
    text-align: left;
}

.header-table th {
    background-color: #222;
    color: white;
}

.header-scroll-box {
    max-height: 300px;
    max-width: 100%;
    overflow: auto;
    background-color: #111;
    margin-bottom: 20px;
}

</style>
"""))

cutout_dir = Path(
    r"C:\Users\dapur\Downloads\Other\Research\rcb_Star_Dust_Survey\test_CNN\data\V_CrA\cutouts"
)
manifest_path = cutout_dir.parent / "plate_manifest.csv"

cutouts = sorted(cutout_dir.glob("*.fits"))
print(f"Found {len(cutouts)} cutouts")

# SPEED FIX : instead of calling fits.getheader() on every single cutout file
# (10,000+ individual disk reads) just to get a date for the dropdown label,
# pull dates from plate_manifest.csv in ONE read. That data was already
# fetched by Cell 1 (obs_date column, from sess.exposures()).
date_lookup = {}
if manifest_path.exists():
    manifest_df_for_dates = pd.read_csv(manifest_path)
    if "filename" in manifest_df_for_dates.columns and "obs_date" in manifest_df_for_dates.columns:
        for _, row in manifest_df_for_dates.iterrows():
            fname = row.get("filename")
            odate = row.get("obs_date")
            if isinstance(fname, str) and fname and pd.notna(odate):
                date_lookup[Path(fname).name] = str(odate)
        print(f"Loaded {len(date_lookup)} dates from manifest (fast path).")
    else:
        print("Manifest found but missing 'filename' or 'obs_date' column -- falling back to per-file header reads.")
else:
    print("No plate_manifest.csv found next to cutouts -- falling back to per-file header reads (slower).")


# Fallback only : pulls an observation date out of a single FITS header.
# Only called for files NOT found in the manifest lookup above.
def get_plate_date_from_header(path):
    try:
        hdr = fits.getheader(path)
    except Exception:
        return path.stem

    for key in ("DATE-OBS", "DATE_OBS", "DATEORIG"):
        if key in hdr and hdr[key]:
            return str(hdr[key])

    for key in ("MJD",):
        if key in hdr and hdr[key]:
            try:
                return Time(float(hdr[key]), format="mjd").iso.split(" ")[0]
            except Exception:
                pass

    for key in ("JD",):
        if key in hdr and hdr[key]:
            try:
                return Time(float(hdr[key]), format="jd").iso.split(" ")[0]
            except Exception:
                pass

    return path.stem


def get_plate_date(path):
    """Fast path: look up date from manifest dict (no disk I/O).
    Slow path (fallback): read it from the FITS header directly."""
    cached = date_lookup.get(path.name)
    if cached is not None:
        return cached
    return get_plate_date_from_header(path)


# Build dropdown options as "Date - Filename"
dropdown_options = []

for f in cutouts:
    filename = str(f.name).strip()

    dropdown_options.append(
        (f"{get_plate_date(f)} - {filename}", f)
    )

# Keep the full, unfiltered list separately. dropdown.options will be
# swapped out live by the search box, so we need the original list preserved
# to filter from and to restore when the search box is cleared.
all_dropdown_options = dropdown_options

# Search box, sits above the dropdown
search_box = widgets.Text(
    placeholder="Search by filename or date...",
    layout=widgets.Layout(width="1200px", margin="0 0 6px 0")
)
search_box.add_class("dark-search")

search_container = widgets.HBox(
    [search_box],
    layout=widgets.Layout(justify_content="center", width="100%")
)

dropdown = widgets.Select(
    options=dropdown_options,
    rows=12,
    description="",
    layout=widgets.Layout(width="1200px", height="300px")
)

dropdown.add_class("dark-dropdown")

dropdown_container = widgets.HBox(
    [dropdown],
    layout=widgets.Layout(
        justify_content="center",
        width="100%"
    )
)


# Toggles for header + pixel data
toggle_header = widgets.ToggleButton(
    value=False,
    description="Show Header",
    layout=widgets.Layout(width="160px")
)

toggle_data = widgets.ToggleButton(
    value=False,
    description="Show Pixel Data [SLOW]",
    layout=widgets.Layout(width="180px")
)

toggle_header.add_class("dark-toggle")
toggle_data.add_class("dark-toggle")


def make_label_handler(toggle, label):
    def handler(change):
        if change["new"]:
            toggle.description = f"Hide {label}"
            toggle.add_class("dark-toggle-active")
        else:
            toggle.description = f"Show {label}"
            toggle.remove_class("dark-toggle-active")
    return handler


toggle_header.observe(make_label_handler(toggle_header, "Header"), names="value")
toggle_data.observe(make_label_handler(toggle_data, "Pixel Data"), names="value")


# Copy Filename button + status label

copy_button = widgets.Button(
    description="Copy Filename",
    layout=widgets.Layout(width="180px")
)
copy_button.add_class("dark-toggle")

copy_status_label = widgets.HTML(value="")

def copy_to_clipboard(text):
    """Injects a small JS snippet into the cell output to write `text` to
    the system clipboard via navigator.clipboard.writeText(). Works in
    VS Code notebooks and JupyterLab since both execute injected <script>
    output client-side."""
    escaped = text.replace("\\", "\\\\").replace('"', '\\"')
    display(Javascript(f'''
        navigator.clipboard.writeText("{escaped}").catch(function(err) {{
            console.error("Clipboard copy failed:", err);
        }});
    '''))

def on_copy_clicked(b):
    if current_path is None:
        copy_status_label.value = "<span style='color:#EF5350;'>No plate selected yet.</span>"
        return
    filename = current_path.name
    copy_to_clipboard(filename)
    copy_status_label.value = f"<span style='color:#66bb6a;'>✅ Copied: {filename}</span>"

copy_button.on_click(on_copy_clicked)

toggles_container = widgets.HBox(
    [toggle_header, toggle_data, copy_button, copy_status_label],
    layout=widgets.Layout(
        justify_content="center",
        width="100%",
        margin="10px 0px"
    )
)


output = widgets.Output(
    layout=widgets.Layout(
        padding="10px",
        width="100%",
        display="flex",
        justify_content="center",
        align_items="center"
    )
)

output.add_class("dark-output")

output_container = widgets.HBox(
    [output],
    layout=widgets.Layout(
        justify_content="center",
        width="100%"
    )
)

headers_output = widgets.Output(layout=widgets.Layout(padding="10px", width="100%"))
headers_output.add_class("dark-output")

headers_container = widgets.HBox(
    [headers_output],
    layout=widgets.Layout(justify_content="center", width="100%")
)

data_output = widgets.Output(layout=widgets.Layout(padding="10px", width="100%"))
data_output.add_class("dark-output")

data_container = widgets.HBox(
    [data_output],
    layout=widgets.Layout(justify_content="center", width="100%")
)

current_path = None
current_data = None

# Filters the dropdown's options live as the user types in search_box.
# Matches against the visible label (date + filename), case-insensitive.
def on_search_change(change):
    query = change["new"].strip().lower()

    if not query:
        dropdown.options = all_dropdown_options
        return

    filtered = [
        (label, path) for label, path in all_dropdown_options
        if query in label.lower()
    ]

    if filtered:
        dropdown.options = filtered
    else:
        # No matches, so show a single disabled-looking placeholder entry
        # instead of an empty/broken dropdown.
        dropdown.options = [("No matches", None)]

search_box.observe(on_search_change, names="value")


def render_header():
    with headers_output:
        clear_output(wait=True)

        if not toggle_header.value or current_path is None:
            return

        header = fits.getheader(current_path)

        header_df = pd.DataFrame(list(header.items()), columns=['Keyword', 'Value'])

        header_html = header_df.to_html(
            classes="header-table",
            border=0,
            index=False
        )

        display(HTML("<h3 style='color:white;text-align:center;'>FITS Header</h3>"))
        display(HTML(f'<div class="header-scroll-box">{header_html}</div>'))


def render_data():
    with data_output:
        clear_output(wait=True)

        if not toggle_data.value or current_data is None:
            return

        df = pd.DataFrame(current_data)
        table_html = df.to_html(classes="dark-table", border=0)

        display(HTML("<h3 style='color:white;text-align:center;'>Pixel Data</h3>"))
        display(HTML(f'<div class="table-scroll-box">{table_html}</div>'))


def show_plate(change):
    global current_path, current_data

    # Guard against the "No matches" placeholder entry, whose value is None
    if change["new"] is None:
        return

    current_path = change["new"]
    current_data = fits.getdata(current_path)

    copy_status_label.value = ""  # Clear old "Copied" message when switching plates

    with output:
        clear_output(wait=True)

        fig = plt.figure(figsize=(12, 10), facecolor="#111111")

        gs = gridspec.GridSpec(1, 3, width_ratios=[0.05, 1.0, 0.05], wspace=0.03)

        ax_spacer = fig.add_subplot(gs[0])
        ax = fig.add_subplot(gs[1])
        cax = fig.add_subplot(gs[2])

        ax_spacer.axis("off")

        fig.patch.set_facecolor("#111111")
        ax.set_facecolor("#111111")

        # INVERTED: "gray_r" is the reverse of "gray" -- flips which end of
        # the colormap is dark vs. light, so the image tones are inverted.
        im = ax.imshow(current_data, origin="lower", cmap="gray_r")

        cbar = fig.colorbar(im, cax=cax)
        cbar.set_label("Plate Intensity", color="white", fontsize=14)
        cbar.ax.tick_params(colors="white")

        ax.set_title(current_path.stem, fontsize=28, color="white", pad=16)
        ax.set_xlabel("X Pixel", color="white")
        ax.set_ylabel("Y Pixel", color="white")
        ax.tick_params(colors="white")

        buf = BytesIO()
        fig.savefig(buf, format="png", facecolor=fig.get_facecolor(), bbox_inches="tight")
        plt.close(fig)
        buf.seek(0)

        display(widgets.Image(value=buf.read(), format="png"))

    render_header()
    render_data()


dropdown.observe(show_plate, names="value")
toggle_header.observe(lambda change: render_header(), names="value")
toggle_data.observe(lambda change: render_data(), names="value")


display(
    search_container,
    dropdown_container,
    toggles_container,
    output_container,
    headers_container,
    data_container
)

# Show first plate automatically
if cutouts:
    show_plate({"new": cutouts[0]})

Found 5402 cutouts
Loaded 5402 dates from manifest (fast path).


In [ ]:
# NOTE : Everything past here is testing